In [1]:
using LowLevelFEM
import LowLevelFEM as L
using LinearAlgebra

In [2]:
structured_rect_mesh()
mat = Material("body")
Pu = Problem([mat], type=:VectorField, dim=2)

Problem("structured_rect", :VectorField, 2, 2, Material[Material("body", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 121, LowLevelFEM.Geometry("", "", 0, 0, nothing, nothing, nothing, nothing), :unknown, :rhs)

In [3]:
#export ChainSum, TransposedChain, withgauss, reduced, full

import Base: +, adjoint, transpose
import LowLevelFEM: ⋅

In [4]:
"""
    ChainSumTerm(chain, gauss)

Internal storage for one term of a compound operator expression.

`chain` is usually an `OpApplied` or a `MatrixChain`.
`gauss` stores the preferred integration rule for this term.
"""
struct ChainSumTerm
    chain::Any
    gauss::Any
end


ChainSumTerm

In [5]:
"""
    ChainSum(terms)

Symbolic sum of operator chains.

Example:

```julia
B = Grad(Pu) ⋅ A + Pu ⋅ G
```

stores two terms without assembling anything.
"""
struct ChainSum
    terms::Vector{ChainSumTerm}
end


ChainSum

In [6]:
"""
TransposedChain(chain)

Symbolic transpose marker for compound expressions.

This does not assemble or numerically transpose anything immediately.
It is interpreted during expansion of the bilinear form.
"""
struct TransposedChain
    chain::Any
end


TransposedChain

In [7]:
"""
CompoundChain(left, mats)

Temporary object representing

```
left ⋅ M1 ⋅ M2 ⋯
```

before the right-hand side is supplied.
"""
struct CompoundChain
    left::Any
    mats::Vector{Any}
end


CompoundChain

In [8]:
"""
CompoundBilinear(left, mats, right)

Temporary object representing

```
left ⋅ M1 ⋅ M2 ⋯ ⋅ right
```

where `left` and/or `right` may be `ChainSum`.
"""
struct CompoundBilinear
    left::Any
    mats::Vector{Any}
    right::Any
end


CompoundBilinear

In [9]:
withgauss(chain, gauss) = ChainSumTerm(chain, gauss)
reduced(chain) = withgauss(chain, :reduced)
full(chain) = withgauss(chain, :full)

_as_sumterm(t::ChainSumTerm) = t
_as_sumterm(x) = ChainSumTerm(x, :full)

_chain_sum(xs...) = ChainSum([_as_sumterm(x) for x in xs])

_is_chain_object(x) =
    x isa L.OpApplied ||
    x isa L.MatrixChain ||
    x isa ChainSum ||
    x isa ChainSumTerm ||
    x isa TransposedChain

function _coeff_mats(c)
    if c isa AbstractMatrix
        _check_coeff_matrix(c)
        return Any[c]
    elseif c isa TensorField
        return Any[tensorfield_to_matrix(c)]
    elseif c isa Number || c isa ScalarField
        return Any[c]
    else
        error("Compound operator: invalid coefficient type $(typeof(c)).")
    end
end

function _op_and_mats(x)
    if x isa L.OpApplied
        return x, Any[]
    elseif x isa L.MatrixChain
        return x.a, copy(x.mats)
    else
        error("Compound operator term must be OpApplied or MatrixChain, got $(typeof(x)).")
    end
end

struct _SideTerm
    chain::Any
    transposed::Bool
    gauss::Any
end

function _side_terms(x; transposed=false)
    if x isa TransposedChain
        return _side_terms(x.chain; transposed=(!transposed))
    elseif x isa ChainSum
        return [_SideTerm(t.chain, transposed, t.gauss) for t in x.terms]

    elseif x isa ChainSumTerm
        return [_SideTerm(x.chain, transposed, x.gauss)]

    elseif x isa L.OpApplied || x isa L.MatrixChain
        return [_SideTerm(x, transposed, :full)]

    else
        error("Compound operator: invalid side type $(typeof(x)).")
    end
end

function _maybe_transpose_mats(mats, transposed::Bool)
    if transposed
        return Any[adjoint(M) for M in reverse(mats)]
    else
        return copy(mats)
    end
end

function _select_gauss(left::_SideTerm, right::_SideTerm, gauss)
    gauss !== :auto && return gauss
    if left.gauss == right.gauss
        return left.gauss
    end

    return :full
end

function _make_bilinear_term(left::_SideTerm, middle::Vector{Any}, right::_SideTerm)
    left_op, left_mats = _op_and_mats(left.chain)
    right_op, right_mats = _op_and_mats(right.chain)
    mats = Any[
        #_maybe_transpose_mats(left_mats, left.transposed)...,
        left_mats...,
        middle...,
        #_maybe_transpose_mats(right_mats, right.transposed)...
        adjoint.(reverse(right_mats))...
    ]

    coef = isempty(mats) ? 1.0 : mats

    #@show left_op
    #@show left_mats

    #@show right_op
    #@show right_mats

    #@show coef
    return L.BilinearTerm(left_op, coef, right_op)
end

function _coeff_mats(c)
    if c isa AbstractMatrix
        return Any[c]
    elseif c isa Number || c isa ScalarField || c isa TensorField
        return Any[c]
    else
        error("Compound operator: invalid coefficient type $(typeof(c)).")
    end
end

_coeff_mats (generic function with 1 method)

In [10]:
# ---------------------------------------------------------------------------

# Addition: build ChainSum

# ---------------------------------------------------------------------------

+(a::ChainSum, b::ChainSum) =
    ChainSum([a.terms...; b.terms...])

+(a::ChainSum, b::Union{L.OpApplied,L.MatrixChain,ChainSumTerm,TransposedChain}) =
    ChainSum([a.terms...; _as_sumterm(b)])

+(a::Union{L.OpApplied,L.MatrixChain,ChainSumTerm,TransposedChain}, b::ChainSum) =
    ChainSum([_as_sumterm(a); b.terms...])

+(a::Union{L.OpApplied,L.MatrixChain,ChainSumTerm,TransposedChain},
    b::Union{L.OpApplied,L.MatrixChain,ChainSumTerm,TransposedChain}) =
    _chain_sum(a, b)

+ (generic function with 247 methods)

In [11]:
# ---------------------------------------------------------------------------

# Symbolic transpose

# ---------------------------------------------------------------------------

adjoint(x::Union{ChainSum,ChainSumTerm,L.MatrixChain,L.OpApplied}) =
    TransposedChain(x)

transpose(x::Union{ChainSum,ChainSumTerm,L.MatrixChain,L.OpApplied}) =
    TransposedChain(x)

transpose (generic function with 53 methods)

In [12]:
# ---------------------------------------------------------------------------

# Dot products involving ChainSum / TransposedChain

# ---------------------------------------------------------------------------

function ⋅(left::Union{ChainSum,TransposedChain,ChainSumTerm}, c)
    return CompoundChain(left, _coeff_mats(c))
end

function ⋅(cc::CompoundChain, c)
    append!(cc.mats, _coeff_mats(c))
    return cc
end

function ⋅(cc::CompoundChain, right::Union{ChainSum,TransposedChain,ChainSumTerm,L.OpApplied,L.MatrixChain})
    return CompoundBilinear(cc.left, cc.mats, right)
end

function ⋅(left::Union{ChainSum,TransposedChain,ChainSumTerm},
    right::Union{ChainSum,TransposedChain,ChainSumTerm,L.OpApplied,L.MatrixChain})
    return CompoundBilinear(left, Any[], right)
end

function ⋅(left::Union{L.OpApplied,L.MatrixChain},
    right::Union{ChainSum,TransposedChain,ChainSumTerm})
    return CompoundBilinear(left, Any[], right)
end

dot (generic function with 71 methods)

In [13]:
function ⋅(A::AbstractMatrix, op::L.OpApplied)
    return op ⋅ adjoint(A)
end

function ⋅(A::AbstractMatrix, ch::L.MatrixChain)
    return ch ⋅ adjoint(A)
end

function ⋅(A::Union{Number,ScalarField}, op::L.OpApplied)
    return op ⋅ A
end

dot (generic function with 74 methods)

In [14]:
# ---------------------------------------------------------------------------

# Integral wrapper

# ---------------------------------------------------------------------------

function ∫(expr::CompoundBilinear;
    Ω=nothing, Γ=nothing, weight=nothing,
    gauss=:auto)
    #@show typeof(expr.left)
    #@show typeof(expr.right)

    left_terms = _side_terms(expr.left)
    right_terms = _side_terms(expr.right)
    #@show left_terms
    #@show right_terms

    K = nothing

    for LL in left_terms
        for R in right_terms
            bt = _make_bilinear_term(LL, expr.mats, R)
            #@show bt
            g = _select_gauss(LL, R, gauss)

            Ki = L.∫(bt; Ω=Ω, Γ=Γ, weight=weight, gauss=g)
            K = K === nothing ? Ki : K + Ki
        end
    end

    return K
end

∫ (generic function with 1 method)

In [15]:
A = [1 0 0 0; 0 1 0 0]
G = [1 0; 0 1]
M = [1 0; 0 1]

2×2 Matrix{Int64}:
 1  0
 0  1

In [16]:
B = Grad(Pu) ⋅ A + Pu ⋅ G

ChainSum(ChainSumTerm[ChainSumTerm(LowLevelFEM.MatrixChain(LowLevelFEM.OpApplied(Problem("structured_rect", :VectorField, 2, 2, Material[Material("body", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 121, LowLevelFEM.Geometry("", "", 0, 0, nothing, nothing, nothing, nothing), :unknown, :rhs), LowLevelFEM.GradOp()), Any[[1 0 0 0; 0 1 0 0]]), :full), ChainSumTerm(LowLevelFEM.MatrixChain(LowLevelFEM.OpApplied(Problem("structured_rect", :VectorField, 2, 2, Material[Material("body", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 121, LowLevelFEM.Geometry("", "", 0, 0, nothing, nothing, nothing, nothing), :unknown, :rhs), LowLevelFEM.IdOp()), Any[[1 0; 0 1]]), :full)])

In [17]:
B' ⋅ M ⋅ B

CompoundBilinear(TransposedChain(ChainSum(ChainSumTerm[ChainSumTerm(LowLevelFEM.MatrixChain(LowLevelFEM.OpApplied(Problem("structured_rect", :VectorField, 2, 2, Material[Material("body", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 121, LowLevelFEM.Geometry("", "", 0, 0, nothing, nothing, nothing, nothing), :unknown, :rhs), LowLevelFEM.GradOp()), Any[[1 0 0 0; 0 1 0 0]]), :full), ChainSumTerm(LowLevelFEM.MatrixChain(LowLevelFEM.OpApplied(Problem("structured_rect", :VectorField, 2, 2, Material[Material("body", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 121, LowLevelFEM.Geometry("", "", 0, 0, nothing, nothing, nothing, nothing), :unknown, :rhs), LowLevelFEM.IdOp()), Any[[1 0; 0 1]]), :full)])), Any[[1 0; 0 1]], ChainSum(ChainSumTerm[ChainSumTerm(LowLevelFEM.MatrixChain(LowLevelFEM.OpApplied(Problem("struc

In [18]:
B = full(A ⋅ Grad(Pu)) + reduced(G ⋅ Pu)

K = ∫(B' ⋅ M ⋅ B; Ω="body", gauss=:auto)

sparse([1, 2, 9, 10, 79, 80, 81, 82, 1, 2  …  46, 47, 48, 221, 222, 223, 224, 239, 240, 242], [1, 1, 1, 1, 1, 1, 1, 1, 2, 2  …  242, 242, 242, 242, 242, 242, 242, 242, 242, 242], [0.6347509617419078, -0.016666666666666663, -0.16600447681208733, -0.008333333333333335, -0.183084295075241, -0.016666666666666663, -0.33316218985457957, -0.008333333333333333, -0.016666666666666663, 0.0014176284085744752  …  0.0009792017405111657, 0.008333333333333331, 0.00017114347875380918, -0.008333333333333325, 0.00017114347875380885, 5.204170427930421e-18, 0.0013243797091584467, -0.03333333333333332, 0.0009792017405111648, 0.004708263185645283], 242, 242)

In [19]:
B1 = A ⋅ Grad(Pu)
B2 = G ⋅ Pu

K2 =
    L.∫(Grad(Pu) ⋅ A' ⋅ M ⋅ A ⋅ Grad(Pu); Ω="body", gauss=:full) +
    L.∫(Grad(Pu) ⋅ A' ⋅ M ⋅ G ⋅ Pu; Ω="body", gauss=:full) +
    L.∫(Pu ⋅ G' ⋅ M ⋅ A ⋅ Grad(Pu); Ω="body", gauss=:full) +
    L.∫(Pu ⋅ G' ⋅ M ⋅ G ⋅ Pu; Ω="body", gauss=:full)

sparse([1, 2, 9, 10, 79, 80, 81, 82, 1, 2  …  46, 47, 48, 221, 222, 223, 224, 239, 240, 242], [1, 1, 1, 1, 1, 1, 1, 1, 2, 2  …  242, 242, 242, 242, 242, 242, 242, 242, 242, 242], [0.6344444444444444, -0.016666666666666663, -0.166111111111111, -0.008333333333333335, -0.18277777777777784, -0.016666666666666663, -0.3330555555555556, -0.008333333333333333, -0.016666666666666663, 0.0011111111111111105  …  0.001111111111111111, 0.008333333333333331, 0.00027777777777777767, -0.008333333333333325, 0.0002777777777777775, 5.204170427930421e-18, 0.0011111111111111107, -0.03333333333333332, 0.0011111111111111107, 0.004444444444444442], 242, 242)

In [20]:
K1[:, :]

LoadError: UndefVarError: `K1` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [ ]:
K2[:, :]

242×242 SparseArrays.SparseMatrixCSC{Float64, Int64} with 3780 stored entries:
⎡⡱⢎⡀⠀⠒⠀⠀⠶⠀⠀⠤⠀⠀⣉⠀⠠⡄⠀⠀⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠐⠂⠀⠰⎤
⎢⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠉⠀⠀⠁⠀⠀⠃⠀⠀⠇⠀⠀⠆⠀⠀⡆⠀⠀⡄⠀⠀⡀⠀⢀⡀⠀⠀⎥
⎢⠘⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠁⠀⠀⠁⠀⠸⢷⣄⠀⎥
⎢⢠⡄⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢀⠀⠀⢠⠀⠙⢷⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⢀⡀⠀⢠⡄⠀⢠⡄⠀⢰⠀⠀⠸⠀⠀⠘⠀⠀⠘⠀⠀⠈⠀⠀⠀⎥
⎢⠀⠃⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⣠⡾⠃⠀⠈⠁⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⡄⢠⡄⠀⠀⠀⠀⠀⠀⠀⠀⠈⣻⣾⡛⠀⣤⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⡀⠀⠀⠀⠀⠀⠀⠀⢀⣠⡾⠛⠈⠻⣦⡘⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠉⠁⠀⠀⠀⠀⠀⠀⠈⠉⠀⠀⠻⣶⡈⠻⣦⡈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠠⠤⠀⠀⠀⠀⠀⠀⠶⠆⠀⠀⠀⠈⠻⣦⡈⠻⣦⡈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⣀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡈⠻⣦⡈⠻⣦⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠉⠁⠀⠀⠀⠀⠀⠉⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡈⠻⣦⡈⠿⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠠⠄⠀⠀⠀⠀⠐⠒⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠛⣦⡌⠛⣤⡉⠛⣤⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⣀⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣧⠈⠻⣦⠈⠻⣦⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠈⠉⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠻⣦⡀⠻⣦⡀⠻⣦⡀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠤⠄⠀⠀⠀⠒⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠛⣤⡈⠛⣤⡈⠛⣤⡀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⢀⣀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⠈⠻⣦⠘⢻⣦⠀⠀⠀⎥
⎢⠀⠀⠀⠈⠁⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠻⣶⣀⠻⣦⡘⠻⣦⡀⎥
⎢⠰⠀⠀⠰⢶⣆⠀⠒⠂⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠛⣶⡈⠻⣦⡈⠛⎥
⎣⢀⡀⠀⠀⠀⠙⢷⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⠈⠻⣦⎦

In [ ]:
K1 = ∫((B1 + B2)' ⋅ M ⋅ (B1 + B2); Ω="body", gauss=:full)

norm(K1.A - K2.A)

0.0

In [ ]:
B = A ⋅ Grad(Pu) + G ⋅ Pu

K3 = ∫(B' ⋅ M ⋅ B; Ω="body", gauss=:full)

sparse([1, 2, 9, 10, 79, 80, 81, 82, 1, 2  …  46, 47, 48, 221, 222, 223, 224, 239, 240, 242], [1, 1, 1, 1, 1, 1, 1, 1, 2, 2  …  242, 242, 242, 242, 242, 242, 242, 242, 242, 242], [0.6344444444444444, -0.016666666666666663, -0.166111111111111, -0.008333333333333335, -0.18277777777777784, -0.016666666666666663, -0.3330555555555556, -0.008333333333333333, -0.016666666666666663, 0.0011111111111111105  …  0.001111111111111111, 0.008333333333333331, 0.00027777777777777767, -0.008333333333333325, 0.0002777777777777775, 5.204170427930421e-18, 0.0011111111111111107, -0.03333333333333332, 0.0011111111111111107, 0.004444444444444442], 242, 242)

In [ ]:
B = full(A ⋅ Grad(Pu)) + reduced(G ⋅ Pu)

K4 = ∫(B' ⋅ M ⋅ B; Ω="body", gauss=:auto)

sparse([1, 2, 9, 10, 79, 80, 81, 82, 1, 2  …  46, 47, 48, 221, 222, 223, 224, 239, 240, 242], [1, 1, 1, 1, 1, 1, 1, 1, 2, 2  …  242, 242, 242, 242, 242, 242, 242, 242, 242, 242], [0.6347509617419078, -0.016666666666666663, -0.16600447681208733, -0.008333333333333335, -0.183084295075241, -0.016666666666666663, -0.33316218985457957, -0.008333333333333333, -0.016666666666666663, 0.0014176284085744752  …  0.0009792017405111657, 0.008333333333333331, 0.00017114347875380918, -0.008333333333333325, 0.00017114347875380885, 5.204170427930421e-18, 0.0013243797091584467, -0.03333333333333332, 0.0009792017405111648, 0.004708263185645283], 242, 242)

In [ ]:
norm(K3.A - K4.A)

0.0076799246999650055